# Causal attention mechanism

In [1]:
import torch
import torch.nn as nn
import tiktoken
import torch.nn.functional as F

In [2]:
vocab_size = 0
emb_dim = 0
d_out = 0
n_layers = 0
n_blocks = 0
kqv_bias = False

In [3]:
class Embeddings(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, emb_dim)
        self.pos_emb = nn.Embedding(vocab_size, emb_dim)

    def forward(self, x):
        B, T = x.shape
        token = self.token_emb(x)
        pos = self.pos_emb(torch.arange(T, device=x.device))
        return token + pos

In [4]:
class AttentionMechnism(nn.Module):
    def __init__(self):
        super().__init__()
        self.w_query = nn.Linear(emb_dim, d_out, kqv_bias)
        self.w_key = nn.Linear(emb_dim, d_out, kqv_bias)
        self.w_value = nn.Linear(emb_dim, d_out, kqv_bias)

    def forward(self, x):
        Q = self.w_query(x)
        K = self.w_key(x)
        V = self.w_value(x)

        scores = Q @ K.transpose(-2, -1)
        scores = scores / (K.shape[-1] ** 0.5)

        mask = torch.triu(torch.ones_like(scores), diagonal=1)
        scores = scores.masked_fill(mask.bool(), float("-inf"))

        attn = F.softmax(scores, dim=-1)
        return attn @ V

In [5]:
class MultiheadAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.heads = nn.ModuleList([AttentionMechnism() for _ in range(n_layers)])

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

In [6]:
class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks = nn.ModuleList([MultiheadAttention() for _ in range(n_blocks)])

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
            return x

In [7]:
embeddings = Embeddings()


In [11]:
mult = MultiheadAttention()
mult.state_dict()

OrderedDict()

In [ ]:
trans = TransformerBlock()
print(trans.forward(2))

None
